# LE Training (5 params) — Structured Analysis

Train 1 control model (standard cross-entropy / MSE loss) and 4 FTLE-regularized models,
each penalizing a different prefix interval [0, t_end]. Model uses 5 piecewise-constant
parameters (non-autonomous dynamics). After training, run the full structured analysis
pipeline: accuracy + attack table, decision boundaries, FTLE grids, and FTLE-at-attack-
points comparison.

Set `load_models = True` to skip training and load saved models.

In [ ]:
import torch
device = torch.device('cpu')
import matplotlib.pyplot as plt
import numpy as np
import importlib
import os
import time

import models.training
import models.neural_odes
import FTLE as FTLE_module
import plots.plots

%matplotlib inline
%config InlineBackend.figure_formats = ['svg']

importlib.reload(models.training)
importlib.reload(models.neural_odes)
importlib.reload(FTLE_module)
importlib.reload(plots.plots)

from models.training import (
    create_dataloader, doublebackTrainer, FTLE_Trainer,
    save_model, save_history, load_model, load_history
)
from models.neural_odes import NeuralODEvar
from FTLE import LEs_batch, LE_grid
from plots.plots import (
    set_plot_defaults, classification_levelsets,
    classification_levelsets_with_attacks, evaluate_models,
    plot_FTLEs, find_attacks, compute_accuracy
)

set_plot_defaults()
torch.backends.cudnn.deterministic = True

## Parameters

In [ ]:
seed          = 60
cross_entropy = False
output_dim    = 2
data_noise    = 0.05
num_points    = 5000
plotlim       = [-2, 2]
subfolder     = 'LE_training_5p_structured'
load_models   = True

os.makedirs(subfolder, exist_ok=True)

# Model architecture (5 piecewise-constant parameters)
hidden_dim, data_dim = 2, 2
T, time_steps        = 10, 100
num_params           = 5
layers_hidden        = 0
non_linearity        = 'tanh'
architecture         = 'inside'

# FTLE regularization
le_reg        = 20.
le_threshold  = 0.05
num_epochs    = 60

# Regularization intervals
interval_ends  = [3., 5., 7., 10.]
time_intervals = [torch.tensor([0., t_end]) for t_end in interval_ends]
interval_labels = [f'[0, {int(t)}]' for t in interval_ends]

# Analysis
ftle_time_interval = torch.tensor([0., float(T)])
adversarial_budget = 0.3
equi_amount        = 72    # directions for equi attack (72 = 5 deg resolution)
margin             = 0.15
le_density         = 100

torch.manual_seed(seed)
print(f'Subfolder: {subfolder}  |  seed={seed}  |  load_models={load_models}')


## Dataset

In [ ]:
label = 'vector'

dataloader, dataloader_viz = create_dataloader(
    'moons', noise=data_noise, cross_entropy=cross_entropy,
    num_points=num_points, plotlim=plotlim, random_state=seed,
    label=label, filename=subfolder + '/trainingset'
)

# Large eval set for evaluate_models (10 000 points for reliable attack counts)
dataloader_big, _ = create_dataloader(
    'moons', noise=data_noise, num_points=10000, batch_size=64,
    plotlim=plotlim, label=label, cross_entropy=False,
    random_state=seed, filename=None
)
_batches_big = [(X, y) for X, y in dataloader_big]
X_full = torch.cat([X for X, y in _batches_big], dim=0)
y_full = torch.cat([y for X, y in _batches_big], dim=0)

_batches_test = [(X, y) for X, y in dataloader_viz]
X_test = torch.cat([X for X, y in _batches_test], dim=0)
y_test = torch.cat([y for X, y in _batches_test], dim=0)

print(f'Train batches: {len(dataloader)} | Test: {len(X_test)} | Eval: {len(X_full)}')

## Model instantiation

All models are seeded identically so weight initialisation is the same.
The control model uses standard training; each regularized model sees a
different FTLE penalty interval. Dynamics are non-autonomous with
`num_params=5` piecewise-constant subintervals.

In [ ]:
def make_model():
    return NeuralODEvar(
        device, data_dim, hidden_dim,
        output_dim=output_dim, non_linearity=non_linearity,
        architecture=architecture, T=T, time_steps=time_steps,
        num_params=num_params, layers_hidden=layers_hidden,
        cross_entropy=cross_entropy,
    )

torch.manual_seed(seed)
anode_control = make_model()

anodes = []
for _ in time_intervals:
    torch.manual_seed(seed)
    anodes.append(make_model())

all_models = [anode_control] + anodes
all_names  = ['control'] + [f'reg {lbl}' for lbl in interval_labels]
print(f'Created 1 control + {len(anodes)} regularized models')

## Training

When `load_models = False` every model is trained from scratch and saved to disk.
Set `load_models = True` and re-run this cell to skip training.

In [ ]:
if load_models:
    print('Loading saved models and histories...')
    anode_control = load_model(subfolder + '/anode_control.pth')
    anodes = [load_model(subfolder + f'/anode_interval_{i}.pth')
              for i in range(len(time_intervals))]
    all_models = [anode_control] + anodes
    histories_control = load_history(subfolder + '/history_control.json')
    histories_list    = [load_history(subfolder + f'/history_interval_{i}.json')
                         for i in range(len(time_intervals))]
    train_times = load_history(subfolder + '/train_times.json')
    print('All models and histories loaded.')

else:
    train_times = {}

    # --- Control model ---
    print('Training control model...')
    optimizer_control = torch.optim.Adam(anode_control.parameters(), lr=1e-3)
    trainer_control = doublebackTrainer(
        anode_control, optimizer_control, device,
        cross_entropy=cross_entropy, turnpike=False, verbose=True
    )
    t0 = time.process_time()
    trainer_control.train(dataloader, num_epochs)
    train_times['control'] = time.process_time() - t0
    save_model(anode_control, subfolder + '/anode_control.pth')
    save_history(trainer_control.histories, subfolder + '/history_control.json')
    histories_control = trainer_control.histories
    print('Control model saved.')

    # --- Regularized models ---
    trainers      = []
    histories_list = []
    for i, (anode, time_interval) in enumerate(zip(anodes, time_intervals)):
        lbl = interval_labels[i]
        print(f'\nTraining regularized model {i+1}/{len(anodes)}: interval {lbl}')
        trainer = FTLE_Trainer(
            anode, device=device,
            le_reg=le_reg, time_interval=time_interval, le_threshold=le_threshold
        )
        t0 = time.process_time()
        trainer.train(dataloader, num_epochs, le_num_iter=5)
        train_times[lbl] = time.process_time() - t0
        trainers.append(trainer)
        save_model(anode, subfolder + f'/anode_interval_{i}.pth')
        save_history(trainer.histories, subfolder + f'/history_interval_{i}.json')
        histories_list.append(trainer.histories)
        print(f'Model {lbl} saved.')

    save_history(train_times, subfolder + '/train_times.json')
    print('\nAll models trained and saved.')

## Loss curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(histories_control['epoch_loss_history'], label='control', linestyle='--')
for lbl, h in zip(interval_labels, histories_list):
    ax.plot(h['epoch_loss_history'], label=f'reg {lbl}')
ax.set_xlabel('Epoch')
ax.set_ylabel('Total loss')
ax.set_title('Total loss')
ax.legend(fontsize=7)

ax = axes[1]
for lbl, h in zip(interval_labels, histories_list):
    ax.plot(h['epoch_loss_le_history'], label=f'reg {lbl}')
ax.set_xlabel('Epoch')
ax.set_ylabel('FTLE penalty')
ax.set_title('FTLE penalty term')
ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig(subfolder + '/losses.png')
plt.show()

In [ ]:
header = f"{'Model':<22} {'CPU time (s)':>14}"
print(header)
print('-' * len(header))
for name, t in train_times.items():
    print(f'{name:<22} {t:>13.1f}s')

## Analysis

### Accuracy + attack table

In [ ]:
from IPython.display import HTML, display

rows = evaluate_models(
    models=[anode_control] + anodes,
    names=['control'] + [f'reg {lbl}' for lbl in interval_labels],
    X=X_full, y=y_full,
    adversarial_budget=adversarial_budget,
    equi_amount=equi_amount, margin=margin,
    visualize=True, subfolder=subfolder, plotlim=plotlim, print_latex=True,
)

### Decision boundaries

In [ ]:
shared = (
    f'seed={seed}, noise={data_noise}, n={num_points}, '
    f'hidden={hidden_dim}, T={T}, params={num_params}'
)
footnotes = {
    'control': f'control | {shared} | no reg, {num_epochs} epochs',
}
for lbl in interval_labels:
    footnotes[f'reg {lbl}'] = (
        f'reg {lbl} | {shared} | '
        f'le_reg={le_reg}, threshold={le_threshold}, {num_epochs} epochs'
    )

for name, model in zip(all_names, all_models):
    classification_levelsets(
        model, fig_name=f'{subfolder}/levelset_{name}',
        plotlim=plotlim, footnote=footnotes[name]
    )

imgs_html = ''.join(
    f'<figure style="margin:4px"><figcaption style="font-size:10px">{name}</figcaption>'
    f'<img src="{subfolder}/levelset_{name}.png?t={time.time()}" width="280"></figure>'
    for name in all_names
)
display(HTML(f'<div style="display:flex;flex-wrap:wrap">{imgs_html}</div>'))

### FTLE grids

In [ ]:
le_density = 100

if load_models:
    out_control = torch.from_numpy(np.load(subfolder + '/le_grid_control.npy'))
    le_grids = [torch.from_numpy(np.load(subfolder + f'/le_grid_{i}.npy'))
                for i in range(len(time_intervals))]
else:
    out_control, _ = LE_grid(anode_control, x_amount=le_density,
                              time_interval=ftle_time_interval)
    np.save(subfolder + '/le_grid_control.npy', out_control.numpy())
    print('Saved control FTLE grid')

    le_grids = []
    for i, (anode, lbl) in enumerate(zip(anodes, interval_labels)):
        out_max, _ = LE_grid(anode, x_amount=le_density, time_interval=ftle_time_interval)
        np.save(subfolder + f'/le_grid_{i}.npy', out_max.numpy())
        le_grids.append(out_max)
        print(f'Saved FTLE grid for reg {lbl}')

In [ ]:
vmin = float(out_control.min())
vmax = 0.25
print(f'Colour scale: vmin={vmin:.3f}, vmax={vmax:.3f}')

plot_FTLEs(out_control, filename=subfolder + '/ftle_control', vmin=vmin, vmax=vmax)
for i, (out_max, lbl) in enumerate(zip(le_grids, interval_labels)):
    plot_FTLEs(out_max, filename=subfolder + f'/ftle_{i}', vmin=vmin, vmax=vmax)

control_html = (
    f'<figure style="margin:4px"><figcaption style="font-size:10px">control</figcaption>'
    f'<img src="{subfolder}/ftle_control.png?t={time.time()}" width="280"></figure>'
)
imgs_html = ''.join(
    f'<figure style="margin:4px"><figcaption style="font-size:10px">reg {lbl}</figcaption>'
    f'<img src="{subfolder}/ftle_{i}.png?t={time.time()}" width="280"></figure>'
    for i, lbl in enumerate(interval_labels)
)
display(HTML(f'<div style="display:flex;flex-wrap:wrap">{control_html}{imgs_html}</div>'))

### FTLE at adversarial attack points

Find high-confidence equidirectional attacks on the **control** model, then evaluate
the max FTLE at those same base points for every model. This shows whether
FTLE regularization reduces sensitivity at the points the attacker exploits.

In [ ]:
X_base, y_base, grad_best, stats = find_attacks(
    anode_control, X_test, y_test,
    adversarial_budget=adversarial_budget, attack_type='equi',
    equi_amount=equi_amount, margin=margin, return_stats=True, verbose=True
)
print(f'High-confidence attack base points: {len(X_base)}')

classification_levelsets_with_attacks(
    anode_control, X_base, y_base, grad=grad_best,
    fig_name=f'{subfolder}/attack_boundary_control',
    plotlim=plotlim, margin=margin
)

In [ ]:
attack_ftles = {}
attack_ftles['control'] = LEs_batch(X_base, anode_control, time_interval=ftle_time_interval)
for anode, lbl in zip(anodes, interval_labels):
    attack_ftles[f'reg {lbl}'] = LEs_batch(X_base, anode, time_interval=ftle_time_interval)

# Report victim point (highest max FTLE under the control model)
victim_idx = attack_ftles['control'][:, 0].argmax().item()
print(f'Victim point index: {victim_idx}')
print(f'Control max FTLE at victim: {attack_ftles["control"][victim_idx, 0].item():.4f}')
print()
print('Mean max FTLE at attack points:')
for name, ftles in attack_ftles.items():
    print(f'  {name:<20}  mean={ftles[:, 0].mean().item():.4f}  std={ftles[:, 0].std().item():.4f}')

In [ ]:
model_labels = list(attack_ftles.keys())
means = [attack_ftles[k][:, 0].mean().item() for k in model_labels]
stds  = [attack_ftles[k][:, 0].std().item()  for k in model_labels]

fig, ax = plt.subplots()
x = np.arange(len(model_labels))
colors = ['C0'] + ['C1'] * len(anodes)
ax.bar(x, means, yerr=stds, capsize=4, color=colors)
ax.set_xticks(x)
ax.set_xticklabels(model_labels, rotation=30, ha='right')
ax.set_ylabel('Mean max FTLE at attack base points')
ax.set_title('FTLE of adversarial base points — control vs regularized')
plt.tight_layout()
plt.savefig(f'{subfolder}/attack_ftle_comparison.png', dpi=200, bbox_inches='tight')
plt.show()

---
## Video: FTLE per subinterval — [0, 10] regularised model

GIF of the finite-time Lyapunov-exponent field for each piecewise-constant
parameter subinterval, for the model regularised over the full window [0, 10]
(`anodes[3]`, the winning interval). Uses `create_gif_subintervals` as in
`LE_training_5params.ipynb`.

In [ ]:


from plots.plots import create_gif_subintervals
from IPython.display import Image, display

# Regularised model trained on the full window [0, 10] (index 3 of interval_ends).
anode_reg_10 = anodes[3]

gif_density  = 50
gif_filename = subfolder + '/FTLE_per_param_reg_[0,10]'
create_gif_subintervals(anode_reg_10, le_density=gif_density,
                        filename=gif_filename, save_pngs=True)
display(Image(filename=gif_filename + '.gif'))

In [ ]:
gif_density  = 50
gif_filename = subfolder + '/FTLE_per_param_control'
create_gif_subintervals(anode_control, le_density=gif_density,
                        filename=gif_filename, save_pngs=True)
display(Image(filename=gif_filename + '.gif'))